In [1]:
# Example OHLC daily data for Google Inc.
from backtesting.test import GOOG

GOOG.tail()

/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

,Open,High,Low,Close,Volume
2013-02-25,802.3,808.41,790.49,790.77,2303900
2013-02-26,795.0,795.95,784.40,790.13,2202500
2013-02-27,794.8,804.75,791.11,799.78,2026100
2013-02-28,801.1,806.99,801.03,801.20,2265800
2013-03-01,797.8,807.14,796.15,806.19,2175400


In [2]:
import pandas as pd


def SMA(values, n):
    """
    Return simple moving average of `values`, at
    each step taking into account `n` previous values.
    """
    return pd.Series(values).rolling(n).mean()

In [3]:
from backtesting import Strategy
from backtesting.lib import crossover


class SmaCross(Strategy):
    # Define the two MA lags as *class variables*
    # for later optimization
    n1 = 10
    n2 = 20
    
    def init(self):
        # Precompute the two moving averages
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)
    
    def next(self):
        # If sma1 crosses above sma2, close any existing
        # short trades, and buy the asset
        if crossover(self.sma1, self.sma2):
            self.position.close()
            self.buy()

        # Else, if sma1 crosses below sma2, close any existing
        # long trades, and sell the asset
        elif crossover(self.sma2, self.sma1):
            self.position.close()
            self.sell()

In [4]:
%%script echo

    def next(self):
        if (self.sma1[-2] < self.sma2[-2] and
                self.sma1[-1] > self.sma2[-1]):
            self.position.close()
            self.buy()

        elif (self.sma1[-2] > self.sma2[-2] and    # Ugh!
              self.sma1[-1] < self.sma2[-1]):
            self.position.close()
            self.sell()

In [5]:
from backtesting import Backtest

bt = Backtest(GOOG, SmaCross, cash=10_000, commission=.002)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2128 [00:00<?, ?bar/s]

/tmp/ipykernel_26227/134679513.py:4: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Start                     2004-08-19 00:00:00
End                       2013-03-01 00:00:00
Duration                   3116 days 00:00:00
Exposure Time [%]                    94.27374
Equity Final [$]                  56263.51934
Equity Peak [$]                   56309.05934
Commissions [$]                   10563.95154
Return [%]                          462.63519
Buy & Hold Return [%]               607.37036
Return (Ann.) [%]                    22.46598
Volatility (Ann.) [%]                 37.4129
CAGR [%]                             14.99343
Sharpe Ratio                          0.60049
Sortino Ratio                          1.1445
Calmar Ratio                           0.6621
Alpha [%]                           450.62135
Beta                                  0.01978
Max. Drawdown [%]                   -33.93159
Avg. Drawdown [%]                    -6.16072
Max. Drawdown Duration      830 days 00:00:00
Avg. Drawdown Duration       50 days 00:00:00
# Trades                          

In [6]:
bt.plot()

GridPlot(id='p1320', ...)

In [7]:
%%time

stats = bt.optimize(n1=range(5, 30, 5),
                    n2=range(10, 70, 5),
                    maximize='Equity Final [$]',
                    constraint=lambda param: param.n1 < param.n2)
stats

Backtest.optimize:   0%|          | 0/50 [00:00<?, ?it/s]

/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of ba

Backtest.run:   0%|          | 0/2133 [00:00<?, ?bar/s]

/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)


Backtest.run:   0%|          | 0/2083 [00:00<?, ?bar/s]

...

/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of ba

Backtest.run:   0%|          | 0/2133 [00:00<?, ?bar/s]

CPU times: user 430 ms, sys: 289 ms, total: 719 ms
Wall time: 1.29 s


/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1545: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = self.run(**dict(zip(heatmap.index.names, best_params)))


Start                     2004-08-19 00:00:00
End                       2013-03-01 00:00:00
Duration                   3116 days 00:00:00
Exposure Time [%]                     98.1378
Equity Final [$]                  77829.05462
Equity Peak [$]                    84982.1921
Commissions [$]                   30771.03746
Return [%]                          678.29055
Buy & Hold Return [%]               687.98749
Return (Ann.) [%]                    27.21765
Volatility (Ann.) [%]                43.21089
CAGR [%]                             18.05088
Sharpe Ratio                          0.62988
Sortino Ratio                         1.27855
Calmar Ratio                          0.61089
Alpha [%]                           614.80095
Beta                                  0.09228
Max. Drawdown [%]                   -44.55387
Avg. Drawdown [%]                    -5.81107
Max. Drawdown Duration     1558 days 00:00:00
Avg. Drawdown Duration       50 days 00:00:00
# Trades                          

In [8]:
stats._strategy

<Strategy SmaCross(n1=10,n2=15)>

In [9]:
bt.plot(plot_volume=False, plot_pl=False)

GridPlot(id='p1566', ...)

In [10]:
stats.tail()

SQN                                                         1.284206
Kelly Criterion                                             0.131964
_strategy                                      SmaCross(n1=10,n2=15)
_equity_curve                       Equity  DrawdownPct DrawdownD...
_trades                 Size  EntryBar  ExitBar  EntryPrice  Exit...
dtype: object

In [11]:
stats['_equity_curve']  # Contains equity/drawdown curves. DrawdownDuration is only defined at ends of DD periods.

,Equity,DrawdownPct,DrawdownDuration
2004-08-19,10000.00000,0.000000,NaT
2004-08-20,10000.00000,0.000000,NaT
2004-08-23,10000.00000,0.000000,NaT
2004-08-24,10000.00000,0.000000,NaT
2004-08-25,10000.00000,0.000000,NaT
...,...,...,...
2013-02-25,76348.73462,0.101591,NaT
2013-02-26,76287.29462,0.102314,NaT
2013-02-27,77213.69462,0.091413,NaT
2013-02-28,77350.01462,0.089809,NaT


In [12]:
stats['_trades']  # Contains individual trade data

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag,"Entry_SMA(C,10)","Exit_SMA(C,10)","Entry_SMA(C,15)","Exit_SMA(C,15)"
0,87,20,60,114.42,185.23,None,None,6108.33090,52.13910,0.613623,2004-09-17,2004-11-12,56 days,None,107.398,181.075,105.751333,183.324000
1,-86,60,69,185.23,175.80,None,None,748.88284,62.09716,0.047011,2004-11-12,2004-11-26,14 days,None,181.075,173.562,183.324000,173.140000
2,95,69,71,175.80,180.71,None,None,398.71310,67.73690,0.023874,2004-11-26,2004-11-30,4 days,None,173.562,173.178,173.140000,174.548667
3,-95,71,75,180.71,179.13,None,None,81.73040,68.36960,0.004761,2004-11-30,2004-12-06,6 days,None,173.178,176.585,174.548667,175.513333
4,96,75,82,179.13,177.99,None,None,-178.00704,68.56704,-0.010351,2004-12-06,2004-12-15,9 days,None,176.585,175.150,175.513333,176.576000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,-90,2056,2085,740.13,687.78,None,None,4454.47620,257.02380,0.066872,2012-10-16,2012-11-29,44 days,None,752.660,667.388,753.988667,664.446667
148,104,2085,2111,687.78,735.54,None,None,4670.98944,296.05056,0.065302,2012-11-29,2013-01-08,40 days,None,667.388,718.499,664.446667,718.996000
149,-103,2111,2113,735.54,742.83,None,None,-1055.41422,304.54422,-0.013931,2013-01-08,2013-01-10,2 days,None,718.499,724.622,718.996000,721.512667
150,101,2113,2121,742.83,735.99,None,None,-989.56164,298.72164,-0.013190,2013-01-10,2013-01-23,13 days,None,724.622,724.316,721.512667,726.406667
